# PINN all methods — epoch-only training and summary export

Denne notebook kører kun epoch-eksperimentet: `10000` Adam-epochs efterfulgt af `2000` LBFGS-iterationer for hvert seed og hver metode.

Den eksporterer stadig kurverne til plotting, men fjerner wall-clock time-eksperimentet. Derudover gemmes de seedvise slutfejl og runtimes, så man direkte kan lave en tabel med median runtime og mean relative L2 ± standardafvigelse over 5 seeds.


Denne version bruger korrekt frosne SoftAdapt-vægte under LBFGS, men bevarer den originale direkte-ratio SoftAdapt-formel.

In [31]:
from pathlib import Path
import time, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Kør i float64. Dette er især vigtigt for finite differences, fordi andenafledte
# beregnes ved differenser af næsten ens netværksoutputs.
dtype = torch.float64
torch.set_default_dtype(dtype)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Torch default dtype:', torch.get_default_dtype())

DATA_DIR = Path('pinn_all_methods_exported_data_epoch_summary')
DATA_DIR.mkdir(exist_ok=True)

# Problemparametre
# PDE: u_t + c*u_y - v*u_yy = 0
# Eksakt løsning: u(y,t)=exp(-v*t)*sin(y-c*t)
v = 1.0   # diffusion/decay-koefficient i den eksakte løsning
c = 1.0   # advektionshastighed
Y = 2.0 * math.pi
T = 2.0

# Træningsparametre
seeds = [1234, 2345, 3456, 4567, 5678]
N_f = 2000          # PDE/collocation-punkter
N_i = 2000          # initial-punkter. Holdes separat fra PDE-punkter.
N_b = 500           # boundary-tider
adam_epochs = 10000
adam_lr = 1e-3
resample_every = 100
eval_every_epochs = 100
lbfgs_max_iter = 2000

# Finite difference-step.
# FD_H_DIV=100 gav i troubleshooting en god balance mellem diskretiseringsfejl og cancellation.
# Hvis du vil teste følsomhed, så prøv fx 50, 100, 200.
FD_H_DIV = 500
h_y = Y / FD_H_DIV
h_t = T / FD_H_DIV
assert 2*h_y < Y and 2*h_t < T, 'FD-step er for stort til domænet.'

# Adaptive refinement
refine_every = 100
n_candidates = 15000
adaptive_gamma = 50.0

# Evalueringsgrid
N_x = 200
N_t = 200

RUN_EPOCH_EXPERIMENT = True
RUN_TIME_EXPERIMENT = False  # fjernet: denne notebook kører ikke tidsforsøget

METHODS = [
    ('baseline', 'Baseline PINN'),
    ('mesh', 'Adaptive refinement'),
    ('finite_difference', 'Finite difference'),
    ('fourier_features', 'Fourier features'),
    ('softadapt', 'SoftAdapt'),
    ('MoE', 'MoE'),
]

Using device: cuda
Torch default dtype: torch.float64


In [32]:
# Fælles evalueringsgrid og eksakt løsning
x = np.linspace(0.0, Y, N_x, dtype=np.float64)
t = np.linspace(0.0, T, N_t, dtype=np.float64)
x_grid, t_grid = np.meshgrid(x, t)
x_tensor = torch.tensor(x_grid.reshape(-1, 1), dtype=dtype, device=device)
t_tensor = torch.tensor(t_grid.reshape(-1, 1), dtype=dtype, device=device)
U_exact = np.exp(-v * t_grid) * np.sin(x_grid - c * t_grid)


def save_array(name, arr):
    np.savetxt(DATA_DIR / f'{name}.txt', np.asarray(arr, dtype=np.float64))


def scale_inputs(y, t):
    return 2.0*y/Y - 1.0, 2.0*t/T - 1.0


def sample_pde_points(method):
    """Sampler PDE/collocation-punkter.

    For finite difference bruges centrale differenser. Derfor skal PDE-punkterne
    ligge i det indre af domænet, så y±h_y og t±h_t ikke evalueres udenfor.
    Dette erstatter den tidligere periodic wrapping i FD-stencilen, som gav
    kunstigt store residualer på utrænede netværk.
    """
    if method == 'finite_difference':
        y_f = h_y + (Y - 2.0*h_y) * torch.rand(N_f, 1, dtype=dtype, device=device)
        t_f = h_t + (T - 2.0*h_t) * torch.rand(N_f, 1, dtype=dtype, device=device)
    else:
        y_f = Y * torch.rand(N_f, 1, dtype=dtype, device=device)
        t_f = T * torch.rand(N_f, 1, dtype=dtype, device=device)
    return y_f, t_f


def sample_initial_points():
    # Initialbetingelsen bør samples uniformt uafhængigt af PDE-punkterne.
    # Ellers får mesh/refinement indirekte en skæv initial-sampling.
    return Y * torch.rand(N_i, 1, dtype=dtype, device=device)


def sample_boundary_times():
    return T * torch.rand(N_b, 1, dtype=dtype, device=device)


def sample_training_points(method):
    y_f, t_f = sample_pde_points(method)
    y_i = sample_initial_points()
    t_b = sample_boundary_times()
    return y_f, t_f, y_i, t_b

In [33]:
class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=64, depth=3, out_dim=1):
        super().__init__()
        layers = []
        last = in_dim
        for _ in range(depth):
            layers += [nn.Linear(last, hidden), nn.Tanh()]
            last = hidden
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, z):
        return self.net(z)

class FlexibleMLP(nn.Module):
    """MLP med valgfri hidden-størrelse per lag.

    Bruges til FourierPINN, så Fourier-modellen får præcis samme antal
    trainable parameters som baseline-netværket.
    """
    def __init__(self, in_dim, hidden_dims, out_dim=1):
        super().__init__()
        layers = []
        last = in_dim
        for hidden in hidden_dims:
            layers += [nn.Linear(last, hidden), nn.Tanh()]
            last = hidden
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, z):
        return self.net(z)

class FourierPINN(nn.Module):
    def __init__(self, K=3, hidden_dims=(62, 65, 53)):
        super().__init__()
        self.K = K
        # Input-dim = 2 + 4K. For K=3 er input-dim 14.
        # hidden_dims=(62,65,53) giver præcis 8577 parametre.
        self.net = FlexibleMLP(in_dim=2 + 4*K, hidden_dims=hidden_dims, out_dim=1)
    def forward(self, z):
        y_scaled = z[:, 0:1]
        t_scaled = z[:, 1:2]
        feats = [y_scaled, t_scaled]
        for k in range(1, self.K + 1):
            feats += [torch.sin(math.pi*k*y_scaled), torch.cos(math.pi*k*y_scaled)]
            feats += [torch.sin(math.pi*k*t_scaled), torch.cos(math.pi*k*t_scaled)]
        return self.net(torch.cat(feats, dim=1))

class MoEPINN(nn.Module):
    def __init__(self, n_experts=2, expert_hidden=62, expert_depth=2, gate_hidden=53):
        super().__init__()
        # 2 eksperter med hidden=62 og en gate med hidden=53 giver præcis 8577 parametre.
        self.experts = nn.ModuleList([MLP(2, expert_hidden, expert_depth, 1) for _ in range(n_experts)])
        self.gate = MLP(2, gate_hidden, 1, n_experts)
        self.n_experts = n_experts
        self.last_gate_mean = None
    def forward(self, z):
        weights = torch.softmax(self.gate(z), dim=1)
        vals = torch.cat([expert(z) for expert in self.experts], dim=1)
        self.last_gate_mean = weights.mean(dim=0)
        return torch.sum(weights * vals, dim=1, keepdim=True)

def make_model(method):
    if method == 'fourier_features':
        return FourierPINN().to(device=device, dtype=dtype)
    if method == 'MoE':
        return MoEPINN().to(device=device, dtype=dtype)
    return MLP().to(device=device, dtype=dtype)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Fairness-tjek: alle metoder skal have samme antal trainable parameters.
_parameter_counts = {method: count_parameters(make_model(method)) for method, _ in METHODS}
print('Parameter counts:', _parameter_counts)
assert len(set(_parameter_counts.values())) == 1, f'Metoderne er ikke parameter-matchede: {_parameter_counts}'
TARGET_NUM_PARAMETERS = next(iter(_parameter_counts.values()))
print('Alle metoder har', TARGET_NUM_PARAMETERS, 'trainable parameters.')

def model_forward(model, y, t):
    ys, ts = scale_inputs(y, t)
    return model(torch.cat([ys, ts], dim=1))


# Dtype-sanity check
_tmp_model = make_model('baseline')
assert next(_tmp_model.parameters()).dtype == dtype, 'Modelparametre har forkert dtype.'
del _tmp_model


Parameter counts: {'baseline': 8577, 'mesh': 8577, 'finite_difference': 8577, 'fourier_features': 8577, 'softadapt': 8577, 'MoE': 8577}
Alle metoder har 8577 trainable parameters.


In [ ]:
def pde_residual_autograd(model, y, t):
    y = y.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)
    u = model_forward(model, y, t)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    r = u_t + c*u_y - v*u_yy
    return u, r


def pde_residual_finite_difference(model, y, t):
    y = y.clone().detach()
    t = t.clone().detach()

    # Ekstra sikkerhed: hvis funktionen kaldes med punkter fra hele domænet,
    # flyttes de ind i det gyldige FD-interval. Ved korrekt sampling ændrer
    # dette intet. Clamp er valgt frem for wrapping

    y = torch.clamp(y, h_y, Y - h_y)
    t = torch.clamp(t, h_t, T - h_t)

    u_center = model_forward(model, y, t)
    u_t = (model_forward(model, y, t + h_t) - model_forward(model, y, t - h_t)) / (2.0*h_t)
    u_y = (model_forward(model, y + h_y, t) - model_forward(model, y - h_y, t)) / (2.0*h_y)
    u_yy = (model_forward(model, y + h_y, t) - 2.0*u_center + model_forward(model, y - h_y, t)) / (h_y**2)
    r = u_t + c*u_y - v*u_yy
    return u_center, r


def pde_residual(model, y, t, method):
    if method == 'finite_difference':
        return pde_residual_finite_difference(model, y, t)
    return pde_residual_autograd(model, y, t)


def periodic_boundary_loss(model, t_b):
    t_b = t_b.clone().detach().requires_grad_(True)
    y_left = torch.zeros_like(t_b).requires_grad_(True)
    y_right = torch.full_like(t_b, Y).requires_grad_(True)
    u_left = model_forward(model, y_left, t_b)
    u_right = model_forward(model, y_right, t_b)
    u_left_y = torch.autograd.grad(u_left, y_left, grad_outputs=torch.ones_like(u_left), create_graph=True)[0]
    u_right_y = torch.autograd.grad(u_right, y_right, grad_outputs=torch.ones_like(u_right), create_graph=True)[0]
    return torch.mean((u_left-u_right)**2) + torch.mean((u_left_y-u_right_y)**2)

In [35]:
class SoftAdapt:
    """SoftAdapt with original direct-ratio formula and correctly frozen LBFGS weights.

    The weight formula is unchanged from the original implementation:
        ratios = loss_now / (loss_prev + eps)
        weights = softmax(beta * (ratios - mean(ratios))) * n_terms

    The only LBFGS-specific fix is that the last weights computed during Adam are stored
    in self.last_weights and reused as constants during LBFGS. This avoids recomputing
    SoftAdapt weights inside the LBFGS closure, where the objective should remain fixed.
    """
    def __init__(self, beta=5.0, eps=1e-8, enabled=True):
        self.beta = beta
        self.eps = eps
        self.enabled = enabled
        self.prev = None
        self.last_weights = None

    def _ones(self, names):
        return {name: torch.tensor(1.0, dtype=dtype, device=device) for name in names}

    def __call__(self, losses, update=True):
        names = list(losses.keys())
        if (not self.enabled) or self.prev is None:
            weights = self._ones(names)
        else:
            ratios = torch.stack([losses[name].detach()/(self.prev[name] + self.eps) for name in names])
            raw = torch.softmax(self.beta*(ratios-ratios.mean()), dim=0) * len(names)
            weights = {name: raw[j].detach() for j, name in enumerate(names)}
        if update:
            self.prev = {name: losses[name].detach() for name in names}
            self.last_weights = {name: weights[name].detach() for name in names}
        return weights

    def fixed_weights_for_lbfgs(self, names=('pde', 'ini', 'bc')):
        if self.last_weights is None:
            return self._ones(list(names))
        return {name: self.last_weights[name].detach().clone() for name in names}


def compute_components(model, y_f, t_f, y_i, t_b, method, softadapt=None, update_softadapt=False, fixed_weights=None):
    # PDE loss
    _, r = pde_residual(model, y_f, t_f, method)
    loss_pde = torch.mean(r**2)

    # Initial loss på u(y,0)=sin(y), med separat uniform initial-sampling.
    u_ini_pred = model_forward(model, y_i, torch.zeros_like(y_i))
    loss_ini = torch.mean((u_ini_pred - torch.sin(y_i))**2)

    # Periodisk boundary loss på både u og u_y.
    loss_bc = periodic_boundary_loss(model, t_b)

    # MoE load balancing
    loss_load = torch.tensor(0.0, dtype=dtype, device=device)
    if method == 'MoE' and getattr(model, 'last_gate_mean', None) is not None:
        target = torch.ones_like(model.last_gate_mean) / model.n_experts
        loss_load = torch.mean((model.last_gate_mean - target)**2)

    if method == 'softadapt':
        weighted_losses = {'pde': loss_pde, 'ini': 10.0*loss_ini, 'bc': 10.0*loss_bc}
        if fixed_weights is not None:
            # During LBFGS, keep SoftAdapt weights fixed. Do not call softadapt(...)
            # inside the closure because LBFGS expects a fixed objective function.
            weights = fixed_weights
        else:
            if softadapt is None:
                raise ValueError("method='softadapt' kræver et SoftAdapt-objekt. Brug softadapt=softadapt og update_softadapt=False i LBFGS-closure.")
            weights = softadapt(weighted_losses, update=update_softadapt)
        loss_total = weights['pde']*weighted_losses['pde'] + weights['ini']*weighted_losses['ini'] + weights['bc']*weighted_losses['bc']
    else:
        loss_total = loss_pde + 10.0*loss_ini + 10.0*loss_bc + loss_load
    return loss_total, loss_pde, loss_ini, loss_bc, loss_load


def loss_values(model, y_f, t_f, y_i, t_b, method):
    with torch.enable_grad():
        soft = SoftAdapt(enabled=False) if method == 'softadapt' else None
        _, pde, ini, bc, _ = compute_components(model, y_f, t_f, y_i, t_b, method, softadapt=soft, update_softadapt=False)
    return pde.detach().cpu().item(), ini.detach().cpu().item(), bc.detach().cpu().item()

In [ ]:
def evaluate_on_grid(model):
    model.eval()
    with torch.no_grad():
        u_pred = model_forward(model, x_tensor, t_tensor).detach().cpu().numpy()
    U_pred = u_pred.reshape(N_t, N_x)
    error = U_pred - U_exact
    abs_error = np.abs(error)
    rel_l2 = np.linalg.norm(error.ravel()) / np.linalg.norm(U_exact.ravel())
    sup_error = np.max(abs_error)
    model.train()
    return U_pred, abs_error, rel_l2, sup_error


def adaptive_refine(model, yt_old, method, n_candidates=n_candidates, gamma=adaptive_gamma):
    # Residualbaseret sampling: alle gamle PDE-punkter erstattes.
    # Bruges kun til method='mesh'. Initial- og boundary-punkter holdes separat
    # og samples uniformt
    n_old = yt_old.shape[0]
    y_cand = Y * torch.rand(n_candidates, 1, dtype=dtype, device=device)
    t_cand = T * torch.rand(n_candidates, 1, dtype=dtype, device=device)
    with torch.enable_grad():
        _, r = pde_residual(model, y_cand, t_cand, method)
        r2 = r.detach().reshape(-1)**2
    med = torch.median(r2)
    if med.item() > 0:
        r2 = torch.clamp(r2, min=med.item(), max=(gamma*med).item())
    else:
        r2 = r2 + torch.tensor(1e-12, dtype=dtype, device=device)
    probs = r2 / torch.sum(r2)
    replacement = n_old > n_candidates
    idx = torch.multinomial(probs, num_samples=n_old, replacement=replacement)
    yt_new = torch.cat([y_cand[idx], t_cand[idx]], dim=1).detach()
    yt_new = yt_new[torch.randperm(yt_new.shape[0], device=device)]
    return yt_new[:, 0:1], yt_new[:, 1:2]


def update_training_points(model, y_f, t_f, y_i, t_b, method, epoch):
    if method == 'mesh':
        if epoch % resample_every == 0:
            y_i = sample_initial_points()
            t_b = sample_boundary_times()
        if epoch > 0 and epoch % refine_every == 0:
            y_f, t_f = adaptive_refine(model, torch.cat([y_f, t_f], dim=1), method=method)
        return y_f, t_f, y_i, t_b

    if epoch % resample_every == 0:
        return sample_training_points(method)
    return y_f, t_f, y_i, t_b

In [37]:
# Sanity checks før lang træning
# Disse checks er designet til at fange præcis den fejl, hvor finite-difference-stencilen
# bruger periodic wrapping og derfor afviger voldsomt fra autograd-residualen på et utrænet netværk.

def fd_autograd_residual_comparison(model, n_test=2000):
    y_test, t_test = sample_pde_points('finite_difference')
    with torch.enable_grad():
        _, r_auto = pde_residual_autograd(model, y_test, t_test)
        _, r_fd = pde_residual_finite_difference(model, y_test, t_test)
    diff = (r_fd - r_auto).detach()
    return {
        'mean_abs_auto': r_auto.detach().abs().mean().cpu().item(),
        'mean_abs_fd': r_fd.detach().abs().mean().cpu().item(),
        'mean_abs_diff': diff.abs().mean().cpu().item(),
        'max_abs_diff': diff.abs().max().cpu().item(),
        'rms_diff': torch.sqrt(torch.mean(diff**2)).cpu().item(),
    }


def exact_solution_torch(y, t):
    return torch.exp(torch.tensor(-v, dtype=dtype, device=device) * t) * torch.sin(y - c*t)


def exact_fd_residual_test(n_test=2000):
    y, t = sample_pde_points('finite_difference')
    u = exact_solution_torch(y, t)
    u_t = (exact_solution_torch(y, t + h_t) - exact_solution_torch(y, t - h_t)) / (2.0*h_t)
    u_y = (exact_solution_torch(y + h_y, t) - exact_solution_torch(y - h_y, t)) / (2.0*h_y)
    u_yy = (exact_solution_torch(y + h_y, t) - 2.0*u + exact_solution_torch(y - h_y, t)) / (h_y**2)
    r = u_t + c*u_y - v*u_yy
    return {
        'mean_abs_exact_fd_residual': r.abs().mean().cpu().item(),
        'max_abs_exact_fd_residual': r.abs().max().cpu().item(),
        'rms_exact_fd_residual': torch.sqrt(torch.mean(r**2)).cpu().item(),
    }

# Kør checks med en fixed seed, så tallet er reproducerbart.
torch.manual_seed(1234)
np.random.seed(1234)
_test_model = make_model('baseline')
assert next(_test_model.parameters()).dtype == dtype
fd_cmp = fd_autograd_residual_comparison(_test_model)
exact_cmp = exact_fd_residual_test()
print('FD vs autograd residual på samme utrænede NN:', fd_cmp)
print('FD residual på eksakt løsning:', exact_cmp)

# Tolerancerne er bevidst moderate. De skal fange grove fejl, ikke forbyde små FD-diskretiseringsfejl.
assert fd_cmp['rms_diff'] < 1e-2, 'FD-residualen afviger for meget fra autograd-residualen. Tjek h_y/h_t og FD-stencilen.'
assert exact_cmp['rms_exact_fd_residual'] < 1e-2, 'FD-residualen er for stor på den eksakte løsning. Tjek PDE-fortegn eller FD-formel.'
del _test_model

FD vs autograd residual på samme utrænede NN: {'mean_abs_auto': 0.05331249742776863, 'mean_abs_fd': 0.05331241666660661, 'mean_abs_diff': 8.271015574390361e-08, 'max_abs_diff': 1.743719813124267e-07, 'rms_diff': 9.746557008759787e-08}
FD residual på eksakt løsning: {'mean_abs_exact_fd_residual': 8.949150502952964e-06, 'max_abs_exact_fd_residual': 3.210921786399312e-05, 'rms_exact_fd_residual': 1.1431314825613617e-05}


In [38]:
def _format_mean_std(mean, std):
    return f"{mean:.3e} ± {std:.3e}"


def train_epoch_experiment(method, label):
    """Train one method for fixed optimizer budget and export seed-wise statistics.

    Fixed budget:
        adam_epochs Adam steps + lbfgs_max_iter LBFGS iterations.

    The exported final arrays are the ones needed for a table like:
        Method | Mean relative L2 ± Std | Median runtime
    """
    print('\n' + '='*80)
    print(f'Epoch experiment: {label}')
    print('='*80)

    epoch_eval_points = np.arange(0, adam_epochs + 1, eval_every_epochs, dtype=float)
    epoch_eval_points = np.append(epoch_eval_points, adam_epochs + lbfgs_max_iter)
    n_eval = len(epoch_eval_points)

    l2_errors = np.full((len(seeds), n_eval), np.nan)
    sup_errors = np.full((len(seeds), n_eval), np.nan)
    loss_pde_hist = np.full((len(seeds), n_eval), np.nan)
    loss_ini_hist = np.full((len(seeds), n_eval), np.nan)
    loss_bc_hist = np.full((len(seeds), n_eval), np.nan)

    # Seed-wise final data for the summary table
    final_l2 = np.full(len(seeds), np.nan)
    final_sup = np.full(len(seeds), np.nan)
    adam_runtime = np.full(len(seeds), np.nan)
    lbfgs_runtime = np.full(len(seeds), np.nan)
    total_runtime = np.full(len(seeds), np.nan)
    lbfgs_closure_calls = np.full(len(seeds), np.nan)

    # Heatmap / mean solution data
    surface = np.full((N_t, N_x, len(seeds)), np.nan)
    pinn_solutions = np.full((N_t, N_x, len(seeds)), np.nan)

    for i, seed in enumerate(seeds):
        print(f'Training {label} with seed = {seed}')
        torch.manual_seed(seed)
        np.random.seed(seed)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        seed_start = time.perf_counter()

        model = make_model(method)
        optimizer = torch.optim.Adam(model.parameters(), lr=adam_lr)
        softadapt = SoftAdapt(beta=5.0, enabled=(method == 'softadapt'))
        y_f, t_f, y_i, t_b = sample_training_points(method)
        eval_idx = 0

        # ---------------------------
        # Adam phase
        # ---------------------------
        if device.type == 'cuda':
            torch.cuda.synchronize()
        adam_start = time.perf_counter()

        for epoch in range(adam_epochs + 1):
            if epoch % eval_every_epochs == 0:
                U_pred, abs_error, rel_l2, sup_error = evaluate_on_grid(model)
                l2_errors[i, eval_idx] = rel_l2
                sup_errors[i, eval_idx] = sup_error
                pde_v, ini_v, bc_v = loss_values(model, y_f, t_f, y_i, t_b, method)
                loss_pde_hist[i, eval_idx] = pde_v
                loss_ini_hist[i, eval_idx] = ini_v
                loss_bc_hist[i, eval_idx] = bc_v
                eval_idx += 1

            if epoch == adam_epochs:
                break

            y_f, t_f, y_i, t_b = update_training_points(model, y_f, t_f, y_i, t_b, method, epoch)
            optimizer.zero_grad(set_to_none=True)
            loss_total, _, _, _, _ = compute_components(
                model, y_f, t_f, y_i, t_b, method,
                softadapt=softadapt,
                update_softadapt=True
            )
            loss_total.backward()
            optimizer.step()

        if device.type == 'cuda':
            torch.cuda.synchronize()
        adam_runtime[i] = time.perf_counter() - adam_start

        # ---------------------------
        # LBFGS phase
        # ---------------------------
        # Bruger nye/fair punkter til LBFGS-polish, ligesom tidligere kode.
        if method == 'mesh':
            y_f, t_f = adaptive_refine(model, torch.cat([y_f, t_f], dim=1), method=method)
            y_i = sample_initial_points()
            t_b = sample_boundary_times()
        else:
            y_f, t_f, y_i, t_b = sample_training_points(method)

        lbfgs = torch.optim.LBFGS(
            model.parameters(), lr=1.0, max_iter=lbfgs_max_iter, max_eval=lbfgs_max_iter,
            tolerance_grad=1e-9, tolerance_change=1e-12, history_size=50, line_search_fn='strong_wolfe'
        )

        closure_counter = {'n': 0}

        # SoftAdapt-specific LBFGS fix:
        # freeze the last weights from the Adam phase and reuse them as constants
        # throughout all LBFGS closure calls. This preserves the original SoftAdapt
        # direct-ratio formula, but prevents the objective from changing inside LBFGS.
        fixed_softadapt_weights = None
        if method == 'softadapt':
            fixed_softadapt_weights = softadapt.fixed_weights_for_lbfgs()
            fixed_softadapt_weights = {
                k: v.detach().to(device=device, dtype=dtype)
                for k, v in fixed_softadapt_weights.items()
            }
            print(
                'SoftAdapt LBFGS fixed weights | '
                f"pde={fixed_softadapt_weights['pde'].item():.3e} | "
                f"ini={fixed_softadapt_weights['ini'].item():.3e} | "
                f"bc={fixed_softadapt_weights['bc'].item():.3e}"
            )

        def closure():
            closure_counter['n'] += 1
            lbfgs.zero_grad(set_to_none=True)
            loss_total, _, _, _, _ = compute_components(
                model, y_f, t_f, y_i, t_b, method,
                softadapt=softadapt,
                update_softadapt=False,
                fixed_weights=fixed_softadapt_weights
            )
            loss_total.backward()
            return loss_total

        if device.type == 'cuda':
            torch.cuda.synchronize()
        lbfgs_start = time.perf_counter()
        lbfgs.step(closure)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        lbfgs_runtime[i] = time.perf_counter() - lbfgs_start
        lbfgs_closure_calls[i] = closure_counter['n']
        total_runtime[i] = time.perf_counter() - seed_start

        # Final evaluation after Adam + LBFGS
        U_pred, abs_error, rel_l2, sup_error = evaluate_on_grid(model)
        final_l2[i] = rel_l2
        final_sup[i] = sup_error

        if eval_idx < n_eval:
            l2_errors[i, eval_idx] = rel_l2
            sup_errors[i, eval_idx] = sup_error
            pde_v, ini_v, bc_v = loss_values(model, y_f, t_f, y_i, t_b, method)
            loss_pde_hist[i, eval_idx] = pde_v
            loss_ini_hist[i, eval_idx] = ini_v
            loss_bc_hist[i, eval_idx] = bc_v

        pinn_solutions[:, :, i] = U_pred
        surface[:, :, i] = abs_error

        print(
            f"seed={seed} | final rel L2={rel_l2:.4e} | sup={sup_error:.4e} | "
            f"Adam={adam_runtime[i]:.1f}s | LBFGS={lbfgs_runtime[i]:.1f}s | "
            f"total={total_runtime[i]:.1f}s | closure_calls={closure_counter['n']}"
        )

    # Existing plotting exports
    save_array(f'{method}_x_grid', x_grid)
    save_array(f'{method}_t_grid', t_grid)
    save_array(f'{method}_epochs_eval', epoch_eval_points)
    save_array(f'{method}_l2_error_vs_epochs', l2_errors)
    save_array(f'{method}_infinity_norm_vs_epochs', sup_errors)
    save_array(f'{method}_loss_pde_vs_epochs', loss_pde_hist)
    save_array(f'{method}_loss_ini_vs_epochs', loss_ini_hist)
    save_array(f'{method}_loss_bc_vs_epochs', loss_bc_hist)
    save_array(f'{method}_enviroment_error', np.nanmean(surface, axis=2))
    save_array(f'{method}_pinn_solution_mean', np.nanmean(pinn_solutions, axis=2))

    # New summary exports for tables
    save_array(f'{method}_final_l2_after_epoch_lbfgs', final_l2)
    save_array(f'{method}_final_sup_after_epoch_lbfgs', final_sup)
    save_array(f'{method}_adam_runtime_epoch', adam_runtime)
    save_array(f'{method}_lbfgs_runtime_epoch', lbfgs_runtime)
    save_array(f'{method}_total_runtime_epoch_lbfgs', total_runtime)
    save_array(f'{method}_lbfgs_closure_calls_epoch', lbfgs_closure_calls)

    return {
        'Method': label,
        'Number of seeds': len(seeds),
        'Mean relative L2': float(np.nanmean(final_l2)),
        'Std relative L2': float(np.nanstd(final_l2, ddof=1)),
        'Median runtime [s]': float(np.nanmedian(total_runtime)),
        'Median Adam runtime [s]': float(np.nanmedian(adam_runtime)),
        'Median LBFGS runtime [s]': float(np.nanmedian(lbfgs_runtime)),
        'Min relative L2': float(np.nanmin(final_l2)),
        'Max relative L2': float(np.nanmax(final_l2)),
    }


In [39]:
def _csv_escape(value):
    s = str(value)
    if any(ch in s for ch in [',', '"', '\n']):
        s = '"' + s.replace('"', '""') + '"'
    return s


def write_csv_manual(path, rows, columns):
    with open(path, 'w', encoding='utf-8') as f:
        f.write(','.join(columns) + '\n')
        for row in rows:
            f.write(','.join(_csv_escape(row.get(col, '')) for col in columns) + '\n')


def latex_escape_text(s):
    s = str(s)
    replacements = {
        '&': r'\&', '%': r'\%', '$': r'\$', '#': r'\#',
        '_': r'\_', '{': r'\{', '}': r'\}', '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}', '\\': r'\textbackslash{}',
    }
    return ''.join(replacements.get(ch, ch) for ch in s)


def write_latex_table_manual(path, rows, columns, caption=None, label=None):
    with open(path, 'w', encoding='utf-8') as f:
        f.write('\\begin{table}[htbp]\n')
        f.write('\\centering\n')
        if caption is not None:
            f.write('\\caption{' + latex_escape_text(caption) + '}\n')
        if label is not None:
            f.write('\\label{' + latex_escape_text(label) + '}\n')
        f.write('\\begin{tabular}{' + 'l'*len(columns) + '}\n')
        f.write('\\toprule\n')
        f.write(' & '.join(latex_escape_text(c) for c in columns) + r' \\' + '\n')
        f.write('\\midrule\n')
        for row in rows:
            f.write(' & '.join(str(row.get(c, '')) for c in columns) + r' \\' + '\n')
        f.write('\\bottomrule\n')
        f.write('\\end{tabular}\n')
        f.write('\\end{table}\n')


def build_summary_table(summary_rows):
    """Create and export ranked summary tables without pandas.

    Exports:
      - epoch_lbfgs_summary_statistics.txt/csv
      - epoch_lbfgs_report_table.txt/csv/tex

    The summary is based on the fixed epoch experiment:
      10000 Adam epochs + 2000 LBFGS iterations.
    """
    if len(summary_rows) == 0:
        raise ValueError('summary_rows er tom. Kør træningen først.')

    rows = sorted(summary_rows, key=lambda r: r['Mean relative L2'])

    ranked_rows = []
    for rank, row in enumerate(rows, start=1):
        new_row = dict(row)
        new_row['Rank'] = rank
        new_row['Mean L2 ± Std'] = _format_mean_std(
            new_row['Mean relative L2'],
            new_row['Std relative L2'],
        )
        ranked_rows.append(new_row)

    # Full machine-readable summary
    full_columns = [
        'Rank',
        'Method',
        'Number of seeds',
        'Mean relative L2',
        'Std relative L2',
        'Median runtime [s]',
        'Median Adam runtime [s]',
        'Median LBFGS runtime [s]',
        'Min relative L2',
        'Max relative L2',
        'Mean L2 ± Std',
    ]

    summary_numeric = np.array([
        [
            row['Rank'],
            row['Number of seeds'],
            row['Mean relative L2'],
            row['Std relative L2'],
            row['Median runtime [s]'],
            row['Median Adam runtime [s]'],
            row['Median LBFGS runtime [s]'],
            row['Min relative L2'],
            row['Max relative L2'],
        ]
        for row in ranked_rows
    ], dtype=np.float64)

    np.savetxt(
        DATA_DIR / 'epoch_lbfgs_summary_statistics_numeric.txt',
        summary_numeric,
        header='Rank Number_of_seeds Mean_relative_L2 Std_relative_L2 Median_runtime_s Median_Adam_runtime_s Median_LBFGS_runtime_s Min_relative_L2 Max_relative_L2',
    )
    write_csv_manual(DATA_DIR / 'epoch_lbfgs_summary_statistics.csv', ranked_rows, full_columns)

    with open(DATA_DIR / 'epoch_lbfgs_summary_statistics.txt', 'w', encoding='utf-8') as f:
        f.write('Rank | Method | N | Mean relative L2 | Std relative L2 | Median runtime [s] | Median Adam [s] | Median LBFGS [s] | Min L2 | Max L2\n')
        f.write('-'*150 + '\n')
        for row in ranked_rows:
            f.write(
                f"{row['Rank']:>4} | {row['Method']:<22} | {row['Number of seeds']:>1} | "
                f"{row['Mean relative L2']:.6e} | {row['Std relative L2']:.6e} | "
                f"{row['Median runtime [s]']:.3f} | {row['Median Adam runtime [s]']:.3f} | "
                f"{row['Median LBFGS runtime [s]']:.3f} | {row['Min relative L2']:.6e} | {row['Max relative L2']:.6e}\n"
            )

    # Compact table for report display
    report_rows = []
    for row in ranked_rows:
        report_rows.append({
            'Rank': row['Rank'],
            'Method': latex_escape_text(row['Method']),
            r'Mean $L^2_{rel}$ ± Std': row['Mean L2 ± Std'].replace('e', r'\times 10^{') + '}' if False else row['Mean L2 ± Std'],
            'Median runtime': f"{row['Median runtime [s]']:.1f}s",
        })

    report_columns = ['Rank', 'Method', r'Mean $L^2_{rel}$ ± Std', 'Median runtime']
    write_csv_manual(DATA_DIR / 'epoch_lbfgs_report_table.csv', report_rows, report_columns)

    with open(DATA_DIR / 'epoch_lbfgs_report_table.txt', 'w', encoding='utf-8') as f:
        f.write('Rank | Method | Mean L2_rel ± Std | Median runtime\n')
        f.write('-'*90 + '\n')
        for row in report_rows:
            f.write(f"{row['Rank']:>4} | {row['Method']:<22} | {row[r'Mean $L^2_{rel}$ ± Std']:<24} | {row['Median runtime']}\n")

    write_latex_table_manual(
        DATA_DIR / 'epoch_lbfgs_report_table.tex',
        report_rows,
        report_columns,
        caption='PINN configuration performance ranked by mean relative L2 error across 5 seeds',
        label='tab:pinn_epoch_lbfgs_summary',
    )

    return ranked_rows, report_rows


In [40]:
# Kør alle metoder og eksportér data.
# Denne version kører KUN fixed-epoch eksperimentet:
# 10000 Adam-epochs + 2000 LBFGS-iterationer per seed.
summary_rows = []

for method, label in METHODS:
    if RUN_EPOCH_EXPERIMENT:
        row = train_epoch_experiment(method, label)
        summary_rows.append(row)

summary_table, report_table = build_summary_table(summary_rows)
print('Færdig. Data er gemt i:', DATA_DIR.resolve())
print('\nRangeret report table:')
for row in report_table:
    print(f"{row['Rank']:>2}. {row['Method']:<22} | {row[r'Mean $L^2_{rel}$ ± Std']:<24} | {row['Median runtime']}")



Epoch experiment: Baseline PINN
Training Baseline PINN with seed = 1234
Training Baseline PINN with seed = 2345
Training Baseline PINN with seed = 3456
Training Baseline PINN with seed = 4567
Training Baseline PINN with seed = 5678

Time experiment: Baseline PINN
Training Baseline PINN with seed = 1234
seed=1234 | target=0s | actual=0.0s | epoch=0 | rel L2=1.043e+00
seed=1234 | target=10s | actual=10.0s | epoch=751 | rel L2=8.877e-02
seed=1234 | target=20s | actual=20.0s | epoch=1508 | rel L2=2.592e-02
seed=1234 | target=30s | actual=30.0s | epoch=2259 | rel L2=9.637e-03
seed=1234 | target=40s | actual=40.0s | epoch=3013 | rel L2=5.894e-03
seed=1234 | target=50s | actual=50.0s | epoch=3769 | rel L2=4.876e-03
seed=1234 | target=60s | actual=60.0s | epoch=4512 | rel L2=4.270e-03
seed=1234 | target=70s | actual=70.0s | epoch=5267 | rel L2=1.093e-02
seed=1234 | target=80s | actual=80.0s | epoch=6023 | rel L2=2.975e-03
seed=1234 | target=90s | actual=90.0s | epoch=6772 | rel L2=1.200e-02
s

In [41]:
# Hurtigt tjek af de vigtigste eksporterede filer
important_patterns = [
    '*_final_l2_after_epoch_lbfgs.txt',
    '*_total_runtime_epoch_lbfgs.txt',
    'epoch_lbfgs_summary_statistics.csv',
    'epoch_lbfgs_report_table.tex',
]

for pattern in important_patterns:
    print(f'\n{pattern}')
    for path in sorted(DATA_DIR.glob(pattern)):
        print(' ', path.name)


baseline_actual_time_after_lbfgs.txt
baseline_actual_times.txt
baseline_enviroment_error.txt
baseline_epochs_at_time.txt
baseline_epochs_before_time_lbfgs.txt
baseline_epochs_eval.txt
baseline_infinity_norm_vs_epochs.txt
baseline_l2_error_time.txt
baseline_l2_error_time_after_lbfgs.txt
baseline_l2_error_vs_epochs.txt
baseline_loss_bc_vs_epochs.txt
baseline_loss_ini_vs_epochs.txt
baseline_loss_pde_vs_epochs.txt
baseline_pinn_solution_mean.txt
baseline_supremumsfejl_time.txt
baseline_supremumsfejl_time_after_lbfgs.txt
baseline_t_grid.txt
baseline_time.txt
baseline_x_grid.txt
finite_difference_actual_time_after_lbfgs.txt
finite_difference_actual_times.txt
finite_difference_enviroment_error.txt
finite_difference_epochs_at_time.txt
finite_difference_epochs_before_time_lbfgs.txt
finite_difference_epochs_eval.txt
finite_difference_infinity_norm_vs_epochs.txt
finite_difference_l2_error_time.txt
finite_difference_l2_error_time_after_lbfgs.txt
finite_difference_l2_error_vs_epochs.txt
finite_diff